In [1]:
import requests
from pathlib import Path
import MDAnalysis as mda
from MDAnalysis.lib.util import NamedStream
from io import StringIO
import gemmi
import pymol
from glob import glob

/Users/katehuddleston/miniforge3/envs/oadmet-gen/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
uniprot_ids = {
               "CYP2D6": "P10635", 
               "CYP3A4": "P08684"
              }


In [3]:
# User defined
common_co_crystals = ["GOL", "IMD", "SO4", "EDO", "PO4", "DMS", "CIT"]

exclude_co_crystals = " ".join([f"and not resname {c}" for c in common_co_crystals ])

In [4]:
#Code below will only do one ID at a time. Set interested KEY below.
target = "CYP3A4"

#### **If you already have input data to process, skip to proper section. 
Not all sections should be run. 

Run cells for your needs. 

The following options are available:

- Download and save raw pdb files from RCSB and save processed final files
- Download raw pdb text from RCSB and save processed final files
- Have existing pdb files that you are interested in processing, and save final files

## Query PDB IDs

Get PDB IDs from UNIPROT ID using RCSB PDB search API

In [5]:
def get_pdb_ids(uniprot_id, rows=1000):
    url = "https://search.rcsb.org/rcsbsearch/v2/query?json="
    query = {
        "query": {
            "type": "terminal",
            "service": "text",
            "parameters": {
                "attribute": "rcsb_polymer_entity_container_identifiers.reference_sequence_identifiers.database_accession",
                "operator": "exact_match",
                "value": uniprot_id
            }
        },
        "return_type": "entry",
        "request_options": {
            "paginate": {
                "start": 0,
                "rows": rows  # default 10, change if 1000 ids found 
            }
        }
    }
    response = requests.post(url, json=query)
    response.raise_for_status()
    result = response.json()
    pdb_ids = [entry["identifier"] for entry in result["result_set"]]
    if len(pdb_ids) < rows:
        print(f"Found {len(pdb_ids)} PDB IDs.")
    else:
        print(f"Found {len(pdb_ids)} PDB IDs. Consider changing rows to greater than {rows}.")
    return pdb_ids

In [6]:
pdb_ids = get_pdb_ids(uniprot_ids[target])

Found 119 PDB IDs.


If using IDs, this downloads PDBs using the RCSB API

In [7]:
def get_rcsb_url(pdb_id, fmt="pdb"):
    url = f"https://files.rcsb.org/download/{pdb_id}.{fmt}"
    return requests.get(url)

def write_file(text, file_path):
    with open(file_path, "w") as f:
        f.write(text)

def convert_cif_to_pdb_gemmi(cif_text):
    """Convert mmCIF text to PDB string using gemmi."""
    doc = gemmi.cif.read_string(cif_text)
    structure = gemmi.make_structure_from_block(doc.sole_block())
    return structure.make_pdb_string()


In [8]:
def get_rcsb_pdb(pdb_id, outdir=".", download_initial=False):
    """
    Download a PDB or mmCIF for the given PDB ID.
    Returns the final PDB content as a string. 
    The initial data file is saved, if specified. 
    """
    if download_initial:
        outdir = Path(outdir)
        outdir.mkdir(parents=True, exist_ok=True)
        pdb_path = outdir / f"{pdb_id}_initial.pdb"

    # First try direct PDB download
    pdb_response = get_rcsb_url(pdb_id, fmt="pdb")
    if pdb_response.status_code == 200:
        print(f"Downloading {pdb_id} from RCSB...")
        pdb_text = pdb_response.text
        if download_initial:
            print(f"Saving {pdb_id} from RCSB...")
            write_file(pdb_text, pdb_path)
        return pdb_text

    # Fallback to CIF
    print(f"PDB for {pdb_id} not found. Checking for mmCIF...")
    cif_response = get_rcsb_url(pdb_id, fmt="cif")
    if cif_response.status_code == 200:
        print(f"mmCIF found. Converting to PDB...")
        pdb_text = convert_cif_to_pdb_gemmi(cif_response.text)
        if download_initial:
            write_file(pdb_text, pdb_path)
        return pdb_text

    raise ValueError(f"Neither PDB nor CIF available for {pdb_id}.")

def get_pdb_path(pdb_dir, pdb_id):
    return glob(f"{pdb_dir}//*{pdb_id}*.pdb")[0]



In [9]:
def process_pdb(pdb_id, 
                pdb_text="", 
                outdir=".", 
                exclude_co_crystals="", 
                input_path=None):
    """
    Process the PDB text to remove common co-crystals and
    save the final processed PDB.
    """
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    final_path = outdir / f"{pdb_id}.pdb"

    if input_path:
        pdb = get_pdb_path(input_path, pdb_id)
        print(pdb)
        u = mda.Universe(pdb)
    else:
        u = mda.Universe(NamedStream(StringIO(pdb_text), f"{pdb_id}.pdb"))
    protein_A = u.select_atoms("protein and chainid A")
    others = u.select_atoms(f"chainid A and not protein and not water {exclude_co_crystals}")
    
    assert 'HEM' in set(others.resnames), "HEM group not found in structure."

    combined = protein_A + others
    lig = combined.select_atoms(f"chainid A and not protein and not resname HEM")
    
    if len(lig) > 0:
        lig.residues.resnames = ["LIG"] * len(lig.residues)
        combined = combined + lig

    final_lig_set = set(combined.select_atoms("chainid A and not protein and not water").resnames)
    assert final_lig_set == {"HEM"} or final_lig_set == {"LIG", "HEM"}

    combined.write(final_path)
    return final_path

## Download Initial Files from RCSB and Save Processed Files

In [10]:
input_dir = f"{target}/raw_pdb"
final_dir = f"{target}/final"

In [11]:
for _id in pdb_ids:
    _id = _id.lower()
    print(f"\nProcessing: {_id}")
    pdb_text = get_rcsb_pdb(_id, outdir=input_dir, download_initial=True)
    try:
        processed_path = process_pdb(_id, pdb_text, outdir=final_dir, exclude_co_crystals=exclude_co_crystals)
        print(f"Saved processed PDB: {processed_path}")
    except Exception as e:
        print(f"Failed to process {_id}: {e}")
    


Processing: 1tqn
Saving 1tqn from RCSB...
Saved processed PDB: CYP3A4/final/1tqn.pdb

Processing: 1w0e
Saving 1w0e from RCSB...


/Users/katehuddleston/miniforge3/envs/oadmet-gen/lib/python3.12/site-packages/MDAnalysis/lib/util.py:708: RuntimeWarning: Constructed NamedStream from a NamedStream
  warnings.warn(
/Users/katehuddleston/miniforge3/envs/oadmet-gen/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1154: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn("Found no information for attr: '{}'"


Saved processed PDB: CYP3A4/final/1w0e.pdb

Processing: 1w0f
Saving 1w0f from RCSB...
Saved processed PDB: CYP3A4/final/1w0f.pdb

Processing: 1w0g
Saving 1w0g from RCSB...
Saved processed PDB: CYP3A4/final/1w0g.pdb

Processing: 2j0d
Saving 2j0d from RCSB...
Saved processed PDB: CYP3A4/final/2j0d.pdb

Processing: 2v0m
Saving 2v0m from RCSB...
Saved processed PDB: CYP3A4/final/2v0m.pdb

Processing: 3nxu
Saving 3nxu from RCSB...
Saved processed PDB: CYP3A4/final/3nxu.pdb

Processing: 3tjs
Saving 3tjs from RCSB...
Saved processed PDB: CYP3A4/final/3tjs.pdb

Processing: 3ua1
Saving 3ua1 from RCSB...
Saved processed PDB: CYP3A4/final/3ua1.pdb

Processing: 4d6z
Saving 4d6z from RCSB...
Saved processed PDB: CYP3A4/final/4d6z.pdb

Processing: 4d75
Saving 4d75 from RCSB...
Saved processed PDB: CYP3A4/final/4d75.pdb

Processing: 4d78
Saving 4d78 from RCSB...
Saved processed PDB: CYP3A4/final/4d78.pdb

Processing: 4d7d
Saving 4d7d from RCSB...
Saved processed PDB: CYP3A4/final/4d7d.pdb

Processing

## Only Grab RCSB PDB text and Save Processed Files

In [88]:
final_dir = f"{target}/final"

In [89]:
for _id in pdb_ids:
    _id = _id.lower()
    print(f"\nProcessing: {_id}")
    pdb_text = get_rcsb_pdb(_id)
    try:
        processed_path = process_pdb(_id, pdb_text, outdir=final_dir, exclude_co_crystals=exclude_co_crystals)
        print(f"Saved processed PDB: {processed_path}")
    except Exception as e:
        print(f"Failed to process {_id}: {e}")


Processing: 1tqn
Saved processed PDB: CYP3A4/final/1tqn.pdb

Processing: 1w0e
Saved processed PDB: CYP3A4/final/1w0e.pdb

Processing: 1w0f
Saved processed PDB: CYP3A4/final/1w0f.pdb

Processing: 1w0g
Saved processed PDB: CYP3A4/final/1w0g.pdb

Processing: 2j0d
Saved processed PDB: CYP3A4/final/2j0d.pdb

Processing: 2v0m
Saved processed PDB: CYP3A4/final/2v0m.pdb

Processing: 3nxu
Saved processed PDB: CYP3A4/final/3nxu.pdb

Processing: 3tjs
Saved processed PDB: CYP3A4/final/3tjs.pdb

Processing: 3ua1
Saved processed PDB: CYP3A4/final/3ua1.pdb

Processing: 4d6z
Saved processed PDB: CYP3A4/final/4d6z.pdb

Processing: 4d75
Saved processed PDB: CYP3A4/final/4d75.pdb

Processing: 4d78
Saved processed PDB: CYP3A4/final/4d78.pdb

Processing: 4d7d
Saved processed PDB: CYP3A4/final/4d7d.pdb

Processing: 4i3q
Saved processed PDB: CYP3A4/final/4i3q.pdb

Processing: 4i4g
Saved processed PDB: CYP3A4/final/4i4g.pdb

Processing: 4i4h
Saved processed PDB: CYP3A4/final/4i4h.pdb

Processing: 4k9t
Saved 

## Process from existing PDB

In [90]:
pdb_dir_path = 'PATH/TO/PDB/FILES'
pdb_dir_path = f"{target}/raw_pdb"

for _id in pdb_ids:
    _id = _id.lower()
    try:
        processed_path = process_pdb(_id, pdb_text, outdir=final_dir, exclude_co_crystals=exclude_co_crystals, input_path=pdb_dir_path)
        print(f"Saved processed PDB: {processed_path}")
    except Exception as e:
        print(f"Failed to process {_id}: {e}")

CYP3A4/raw_pdb/1tqn_initial.pdb
Saved processed PDB: CYP3A4/final/1tqn.pdb
CYP3A4/raw_pdb/1w0e_initial.pdb
Saved processed PDB: CYP3A4/final/1w0e.pdb
CYP3A4/raw_pdb/1w0f_initial.pdb
Saved processed PDB: CYP3A4/final/1w0f.pdb
CYP3A4/raw_pdb/1w0g_initial.pdb
Saved processed PDB: CYP3A4/final/1w0g.pdb
CYP3A4/raw_pdb/2j0d_initial.pdb
Saved processed PDB: CYP3A4/final/2j0d.pdb
CYP3A4/raw_pdb/2v0m_initial.pdb
Saved processed PDB: CYP3A4/final/2v0m.pdb
CYP3A4/raw_pdb/3nxu_initial.pdb
Saved processed PDB: CYP3A4/final/3nxu.pdb
CYP3A4/raw_pdb/3tjs_initial.pdb
Saved processed PDB: CYP3A4/final/3tjs.pdb
CYP3A4/raw_pdb/3ua1_initial.pdb
Saved processed PDB: CYP3A4/final/3ua1.pdb
CYP3A4/raw_pdb/4d6z_initial.pdb
Saved processed PDB: CYP3A4/final/4d6z.pdb
CYP3A4/raw_pdb/4d75_initial.pdb
Saved processed PDB: CYP3A4/final/4d75.pdb
CYP3A4/raw_pdb/4d78_initial.pdb
Saved processed PDB: CYP3A4/final/4d78.pdb
CYP3A4/raw_pdb/4d7d_initial.pdb
Saved processed PDB: CYP3A4/final/4d7d.pdb
CYP3A4/raw_pdb/4i3q_initi